In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

In [ ]:
df= pd.read_csv('insurance.csv')
df

##EDA

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
#numerical col so we're using histplot
numeric_columns= ['age', 'bmi', 'children', 'charges']

# Calculate number of rows and columns for subplots
n_cols = 2
n_rows = (len(numeric_columns) + n_cols - 1) // n_cols # Ceiling division

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols)
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, col in enumerate(numeric_columns):
    sns.histplot(df[col], kde=True, ax=axes[i], bins=20)

    axes[i].set_title(f'Distribution of {col.capitalize()}')
    axes[i].set_xlabel(col.capitalize())
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# categorical col so we're using countplot
categorical_columns= ['sex', 'smoker', 'region']
n_cols= 2
n_rows= (len(categorical_columns) + n_cols - 1) // n_cols

fig,axes= plt.subplots(nrows=n_rows, ncols=n_cols)
axes= axes.flatten()

for i, col in enumerate(categorical_columns):
    sns.countplot(data=df, x=col, ax=axes[i])
    axes[i].set_title(f'Count of {col.capitalize()}')
    axes[i].set_xlabel(col.capitalize())
    axes[i].set_ylabel('Count')
    if col == 'region':
        axes[i].tick_params(axis='x', rotation=90)

# Turn off any unused subplots
for j in range(len(categorical_columns), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# numerical and stats col so  we're using boxplot
numerical_stats_columns= ['age', 'bmi', 'children', 'charges']

n_cols=2
n_rows= (len(numerical_stats_columns) + n_cols - 1) // n_cols

fig, axes= plt.subplots(nrows=n_rows, ncols=n_cols)
axes= axes.flatten()

for i, col in enumerate(numerical_stats_columns):
    sns.boxplot(data=df, x=col, ax=axes[i])
    axes[i].set_title(f'Boxplot of {col.capitalize()}')
    axes[i].set_xlabel(col.capitalize())
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()



In [ ]:
# heatmap for co-relation
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.show()

##Data Cleaning and preprocessing

In [ ]:
df_cleaned=df.copy()
df_cleaned

# its very imp to make a copy of og csv if you messed with the original data you have and option to go back to og data

In [ ]:
df_cleaned.drop_duplicates(inplace= True)
df_cleaned.shape

In [ ]:
df_cleaned.isnull().sum()

In [ ]:
df_cleaned.dtypes

In [ ]:
df_cleaned['sex'] = df['sex']
display(df_cleaned.head())

In [ ]:
# label encoding

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['sex'] = df['sex']
df_cleaned['sex'] = df_cleaned['sex'].map({"male" : 0,"female" : 1})
df_cleaned.head()

In [ ]:
df_cleaned['smoker'] = df['smoker']
df_cleaned['smoker'] = df_cleaned['smoker'].map({"no" : 0,"yes" : 1})
df_cleaned.head()

In [ ]:
df_cleaned.rename(columns={
    'sex' : 'is_female',
    'smoker' : 'is_smoker'
},inplace=True)
df_cleaned

In [ ]:
# one-hot encoding

In [ ]:
df_cleaned['region'].value_counts()

In [ ]:
df_cleaned=pd.get_dummies(df_cleaned, columns=['region'], drop_first=True)
df_cleaned

In [ ]:
df_cleaned['bmi'] = df['bmi']
df_cleaned['charges'] = df['charges']

region_cols = ['region_northwest', 'region_southeast', 'region_southwest']
df_cleaned[region_cols] = df_cleaned[region_cols].astype(int)
df_cleaned

##Feature Engineering and Extraction

In [ ]:
sns.histplot(df_cleaned['bmi'])

In [ ]:
df_cleaned['bmi_category'] = pd.cut(
    df_cleaned['bmi'],
    bins=[0, 18.5, 24.9, 29.9, float('inf')],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
    )
df_cleaned

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns = ['bmi_category'],drop_first=True)

In [ ]:
df_cleaned = df_cleaned.astype(int)

In [ ]:

df_cleaned.head()

In [ ]:
df_cleaned.columns


In [ ]:
# StandardScaler tells us how far a value is from the column's average, measured in standard deviations.

In [ ]:
from sklearn.preprocessing import StandardScaler
cols=['age', 'bmi', 'children', 'charges']
scaler=StandardScaler()
df_cleaned[cols]=scaler.fit_transform(df_cleaned[cols])
df_cleaned.head()

In [ ]:
from scipy.stats import pearsonr
# List of features to check against target
selected_features = [
    'age', 'bmi', 'children', 'is_female', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese'
    ]
correlations= {
    feature: pearsonr(df_cleaned[feature], df_cleaned['charges'])[0]
    for feature in selected_features
}
correlation_df = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Pearson Correlation'])
correlation_df.sort_values(by='Pearson Correlation', ascending=False)


In [ ]:

cat_features = [
    'is_female', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese'
]

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

alpha = 0.05

df_cleaned['charges_bin'] = pd.qcut(df_cleaned['charges'], q=4, labels=False)
chi2_results = {}

for col in cat_features:
    contingency = pd.crosstab(df_cleaned[col], df_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = 'Reject Null (Keep Feature)' if p_val < alpha else 'Accept Null (Drop Feature)'
    chi2_results[col] = {
        'chi2_statistic': chi2_stat,
        'p_value': p_val,
        'Decision': decision
    }

chi2_df = pd.DataFrame(chi2_results).T
chi2_df = chi2_df.sort_values(by='p_value')
chi2_df

In [ ]:
final_df = df_cleaned[['age', 'is_female', 'bmi', 'children', 'is_smoker', 'charges','region_southeast','bmi_category_Obese']]

In [ ]:
final_df

In [ ]:
final_df.info()